# Gold Standard Sampling

In [1]:
import pandas as pd

# ===== 경로 설정 =====
INPUT_PATH = "datas/processed/job_postings_processed.csv"
OUTPUT_PATH = "datas/processed/gold_sample_20.csv"

# ===== 데이터 로드 =====
# ✅ job_posting_id를 문자열로 강제 로드(과학표기 방지 핵심)
df = pd.read_csv(INPUT_PATH, dtype={"job_posting_id": "string"})

# ===== 기본 점검 =====
if "job_posting_id" not in df.columns:
    raise ValueError("job_posting_id 컬럼이 필요합니다.")

# ✅ 공백/결측 정리 + 소수점(.0) 제거(혹시 숫자로 들어온 흔적 방지)
df["job_posting_id"] = (
    df["job_posting_id"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0+$", "", regex=True)
)

# ✅ 빈 값 제거(안전)
df = df[df["job_posting_id"].notna() & (df["job_posting_id"] != "")].copy()

# ✅ 중복 제거 (안전장치)
df = df.drop_duplicates(subset=["job_posting_id"])

# ===== 랜덤 샘플링 =====
sample_size = 20
random_seed = 42   # 재현성 확보

if len(df) < sample_size:
    raise ValueError(f"데이터가 {sample_size}개보다 적습니다. 현재 개수: {len(df)}")

sample_df = df.sample(n=sample_size, random_state=random_seed).copy()

# ✅ 저장 전에도 문자열 타입 재확인(엑셀/CSV에서 과학표기 최소화)
sample_df["job_posting_id"] = sample_df["job_posting_id"].astype(str)

# ===== 저장 =====
sample_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print(f"[OK] {sample_size}개 샘플 저장 완료 → {OUTPUT_PATH}")
print(f"샘플 개수: {len(sample_df)}")

[OK] 20개 샘플 저장 완료 → datas/processed/gold_sample_20.csv
샘플 개수: 20
